# 🧪 Lab 6: Housing Price Prediction with LIME – Model Explainability in Action

## 🎯 Objective

This lab introduces the use of **LIME (Local Interpretable Model-agnostic Explanations)** to interpret machine learning models for regression problems. You'll apply LIME to a housing price prediction model and learn how to understand **what features drive a prediction**, and how interpretability improves trust and decision-making in ML systems.

---

## 📦 What You'll Do

- Train a regression model on a housing dataset
- Use LIME to generate explanations for individual predictions
- Visualize and interpret the local contributions of features
- Understand how local interpretability differs from global interpretability
- Learn when LIME explanations may or may not be trustworthy

---

## 🧠 Why This Matters

Most machine learning models—especially tree-based or ensemble models—are **black boxes** by default. Without explanation tools like LIME, it's hard to understand **why** a model makes a certain prediction.

In domains like housing, finance, and healthcare, model transparency is essential for:
- Building user trust
- Complying with regulations (e.g., explainable AI)
- Debugging and improving model fairness

---

## 🛠️ Skills You'll Develop

- Feature contribution analysis
- Local interpretability and trust-building
- Visualization of model behavior using `lime.lime_tabular`
- Critical thinking about explanation reliability and biases

---

Let’s dive into LIME and make the black box a little more transparent! 🔍



### Code Explanation

This line is part of the end-to-end housing price prediction workflow. It either prepares the data, trains the model, evaluates predictions, or interprets results using LIME.

In [1]:
# Install LIME if not already installed
!pip -qqq install lime


### Code Explanation

This block loads the housing dataset into a pandas DataFrame using `read_csv`. It includes variables like area income, number of rooms, and house age. The `.head()` method is used to preview the data and confirm it's correctly loaded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

import lime
import lime.lime_tabular

## 🧭 Step 1: Load and Preprocess the Housing Dataset

We begin by importing and preparing the dataset. In this step, we:

- Load the housing data from a CSV file
- Drop missing values to simplify the analysis
- Separate the target variable (`Price`) from the feature set
- Apply one-hot encoding to categorical variables to make them model-compatible

Using `pd.get_dummies()` allows us to convert categorical columns into binary indicators, which are required by most machine learning algorithms.

---

### 📝 Reflection:
- **Q1:** Why do we use one-hot encoding instead of label encoding here?


In [ ]:
# === Step 1: Load and Preprocess Dataset ===
df = pd.read_csv("housing.csv")
df.dropna(inplace=True)
X = pd.get_dummies(df.drop('Price', axis=1), drop_first=True)
y = df['Price']

## 📊 Step 2: Train/Test Split

We split the dataset into training and testing sets using an 80/20 ratio. This ensures that we evaluate the model on data it has never seen before, which helps us assess generalization performance.

We use `random_state=42` for reproducibility.

---

### ✅ What You’re Learning:
- The importance of data splitting for unbiased model evaluation
- How to prepare features and targets for regression problems


In [ ]:
# === Step 2: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 🤖 Step 3: Train the Random Forest Regressor

We train a `RandomForestRegressor`, a powerful ensemble learning method based on decision trees. It handles non-linearities, works well with mixed feature types, and is robust to overfitting with proper configuration.

After training, we predict prices on the test set and compute the RMSE (Root Mean Squared Error) to measure prediction error.

---

### 📝 Interpretation:
- A lower RMSE indicates better predictive performance.

### 📝 Reflection:
- What factors might influence the RMSE beyond model architecture?


In [ ]:
# === Step 3: Train Model ===
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.2f}")

## 🔍 Step 4: Global Feature Importance

We analyze the overall influence of each feature on the Random Forest model's predictions. The `feature_importances_` attribute measures how much each feature contributed to reducing prediction error across all trees.

We visualize the top 10 features using a bar plot.

---

### 💡 Insight:
- Global feature importance gives an *overall* sense of what matters in the model.
- It does **not** explain **individual predictions**.

### 📝 Reflection:
- **Q2:** Which features had the most impact and why might they be important in predicting house prices?


In [ ]:
# === Step 4: Global Feature Importance ===
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices][:10], y=X.columns[indices][:10])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

## 🧠 Step 5: LIME Explanations – Local Interpretability

We now use **LIME (Local Interpretable Model-agnostic Explanations)** to explain specific predictions made by the model. LIME perturbs the input data locally and fits a simple model (like linear regression) to approximate the complex model near a single point.

This gives us a breakdown of how each feature affected a **specific** prediction.

We explain three individual samples and view the explanation tables inline.

---

### 🧪 Key Concept:
- LIME is **model-agnostic** and gives **local explanations**, which help us understand decisions on a case-by-case basis.

### 📝 Reflection:
- How do LIME explanations differ from global feature importance?


In [ ]:
# === Step 5: LIME Explanation ===
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    mode='regression'
)
# Explain multiple samples
for i in range(3):
    exp = explainer.explain_instance(X_test.values[i], model.predict)
    print(f"\n🔍 LIME Explanation for Sample {i}")
    exp.show_in_notebook(show_table=True)

## 🚨 Step 6: Detect Unusual Predictions with LIME

We loop through multiple predictions and use LIME to identify cases where individual features have an **unusually large influence** (e.g., > $50,000 impact).

This is useful for:
- Debugging models
- Spotting outliers
- Detecting possible bias or instability

---

### 🧠 Thought Experiment:
- **Q3:** How might we use this technique to detect geographic or socioeconomic bias?


In [ ]:
# === Step 6: Detect Unusual Predictions ===
for i in range(10):
    exp = explainer.explain_instance(X_test.values[i], model.predict)
    top_features = dict(exp.as_list())
    if any(abs(v) > 50000 for v in top_features.values()):
        print(f"Sample {i} shows unusually large feature impact")

In [ ]:
# === Embedded Questions ===
# Q1: Why do we use one-hot encoding?
# Q2: Which features have most impact and why?
# Q3: How can we tell if model is making biased predictions?

# === Quiz ===
# 1. What is the main advantage of LIME? (Answer: B - individual explanations)
# 2. What suggests geographic bias? (Answer: B - high weights on lat/lon)
# 3. How can LIME help in debugging? (Answer: B - highlight feature dominance)